# 05. RAG System Testing - Complete Pipeline

This notebook demonstrates and tests the complete RAG (Retrieval-Augmented Generation) system that combines semantic search with LLM answer generation.

## Objectives:
- Test complete RAG pipeline with Gemini LLM integration
- Demonstrate question answering with citations
- Test conversation memory and context handling
- Evaluate answer quality and confidence scoring
- Test FastAPI RAG endpoints
- Analyze system performance and reliability

## RAG Pipeline Features:
- **Retrieval**: Semantic search with document filtering
- **Generation**: Gemini LLM with structured prompts
- **Citations**: Automatic source referencing
- **Confidence**: Answer quality scoring
- **Follow-ups**: Intelligent question suggestions
- **Memory**: Conversation context management

## 1. Environment Setup

In [ ]:
import sys
import os
import json
import time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime
import logging
from tqdm import tqdm
import requests
import warnings
warnings.filterwarnings('ignore')

# Add src to path
project_root = Path.cwd().parent
src_path = project_root / "src"
sys.path.insert(0, str(src_path))

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

print(f"📁 Project root: {project_root}")
print(f"📂 Source path: {src_path}")

In [ ]:
# Import RAG modules
from generation.rag_pipeline import RAGPipeline, create_rag_pipeline
from generation.llm_integration import GeminiLLMGenerator, create_llm_generator
from retrieval.semantic_search import create_semantic_searcher
from config import settings

print("✅ Successfully imported RAG modules")
print(f"🤖 LLM Model: {settings.llm_model}")
print(f"📝 Embedding Model: {settings.embedding_model}")
print(f"📊 Collection: {settings.collection_name}")
print(f"🎯 Top-K Retrieval: {settings.top_k_retrieval}")

## 2. RAG Pipeline Initialization

In [ ]:
# Initialize RAG pipeline
print("🚀 Initializing RAG pipeline...\n")

try:
    rag_pipeline = create_rag_pipeline()
    print("✅ RAG pipeline initialized successfully!")
    
    # Test pipeline health
    health_status = rag_pipeline.health_check()
    print(f"\n🏥 System Health Check:")
    print(f"   Overall health: {health_status['overall_health']}")
    
    for component, status in health_status['components'].items():
        component_status = status.get('status', 'unknown')
        print(f"   {component}: {'✅' if component_status == 'healthy' else '❌'} {component_status}")
        
        if component == 'retrieval' and 'documents' in status:
            print(f"      Documents: {status['documents']:,}")
        elif component == 'generation' and 'model' in status:
            print(f"      Model: {status['model']}")
    
    pipeline_ready = health_status['overall_health'] == 'healthy'
    
except Exception as e:
    print(f"❌ Failed to initialize RAG pipeline: {str(e)}")
    print("\nPlease ensure:")
    print("1. Vector database is populated (run notebook 04_retrieval_demo.ipynb)")
    print("2. Google API key is configured in .env file")
    print("3. Qdrant connection is working")
    pipeline_ready = False
    rag_pipeline = None

## 3. Basic RAG Query Testing

In [ ]:
# Test basic RAG queries
if pipeline_ready:
    print("🧪 Testing basic RAG queries...\n")
    
    # Test queries covering different LangChain topics
    test_queries = [
        {
            "question": "What is LangChain and what are its main components?",
            "expected_topics": ["langchain", "components", "framework", "llm"]
        },
        {
            "question": "How do I create a simple chat application using LangChain?",
            "expected_topics": ["chat", "application", "conversation", "memory"]
        },
        {
            "question": "What are document loaders and how do I use them?",
            "expected_topics": ["document", "loader", "text", "file"]
        },
        {
            "question": "How does vector store integration work in LangChain?",
            "expected_topics": ["vector", "store", "embedding", "similarity"]
        },
        {
            "question": "Can you explain LangChain agents and their tools?",
            "expected_topics": ["agent", "tool", "action", "reasoning"]
        }
    ]
    
    rag_results = []
    
    for i, test_case in enumerate(test_queries, 1):
        question = test_case["question"]
        expected_topics = test_case["expected_topics"]
        
        print(f"🔍 Query {i}: {question}")
        
        start_time = time.time()
        
        try:
            # Execute RAG query
            result = rag_pipeline.query(
                question=question,
                conversation_id=f"test_conversation_{i}"
            )
            
            query_time = (time.time() - start_time) * 1000
            
            if result["success"]:
                answer = result["answer"]
                citations = result["citations"]
                confidence = result["confidence_score"]
                retrieved_docs = result["retrieved_documents"]
                followups = result.get("follow_up_questions", [])
                
                print(f"   ✅ Success (Time: {query_time:.1f}ms)")
                print(f"   📊 Retrieved docs: {retrieved_docs}")
                print(f"   🎯 Confidence: {confidence:.3f}")
                print(f"   📚 Citations: {len(citations)}")
                print(f"   ❓ Follow-ups: {len(followups)}")
                
                # Show answer preview
                answer_preview = answer[:300] + "..." if len(answer) > 300 else answer
                print(f"   💬 Answer preview: {answer_preview}")
                
                # Show citations
                if citations:
                    print(f"   📋 Top citations:")
                    for j, citation in enumerate(citations[:2], 1):
                        source = citation.get('file_name', 'Unknown')
                        score = citation.get('similarity_score', 0)
                        print(f"      {j}. {source} (score: {score:.3f})")
                
                # Show follow-up questions
                if followups:
                    print(f"   ❓ Follow-up questions:")
                    for j, followup in enumerate(followups[:2], 1):
                        print(f"      {j}. {followup}")
                
                # Assess topic coverage
                answer_lower = answer.lower()
                topic_coverage = sum(1 for topic in expected_topics if topic in answer_lower)
                coverage_score = topic_coverage / len(expected_topics)
                
                rag_results.append({
                    'question': question,
                    'success': True,
                    'confidence': confidence,
                    'retrieved_docs': retrieved_docs,
                    'citations_count': len(citations),
                    'followups_count': len(followups),
                    'query_time_ms': query_time,
                    'answer_length': len(answer),
                    'topic_coverage': coverage_score,
                    'retrieval_time': result['performance']['retrieval_time_ms'],
                    'generation_time': result['performance']['generation_time_ms']
                })
                
            else:
                print(f"   ❌ Failed: {result.get('error', 'Unknown error')}")
                rag_results.append({
                    'question': question,
                    'success': False,
                    'confidence': 0,
                    'retrieved_docs': result.get('retrieved_documents', 0),
                    'citations_count': 0,
                    'followups_count': 0,
                    'query_time_ms': query_time,
                    'answer_length': 0,
                    'topic_coverage': 0,
                    'retrieval_time': 0,
                    'generation_time': 0
                })
        
        except Exception as e:
            print(f"   ❌ Error: {str(e)}")
            rag_results.append({
                'question': question,
                'success': False,
                'confidence': 0,
                'retrieved_docs': 0,
                'citations_count': 0,
                'followups_count': 0,
                'query_time_ms': (time.time() - start_time) * 1000,
                'answer_length': 0,
                'topic_coverage': 0,
                'retrieval_time': 0,
                'generation_time': 0
            })
        
        print()
    
    print(f"✅ Completed testing {len(test_queries)} RAG queries")
else:
    print("⚠️  Skipping RAG query tests - pipeline not ready")
    rag_results = []

## 4. Conversation Context Testing

In [ ]:
# Test conversation context and memory
if pipeline_ready:
    print("💬 Testing conversation context and memory...\n")
    
    # Multi-turn conversation scenario
    conversation_id = "test_conversation_memory"
    
    conversation_turns = [
        "What are LangChain chains?",
        "How do I create a simple chain?",
        "Can you show me an example with code?",
        "What about error handling in chains?"
    ]
    
    conversation_results = []
    
    for i, question in enumerate(conversation_turns, 1):
        print(f"🗣️  Turn {i}: {question}")
        
        try:
            result = rag_pipeline.query(
                question=question,
                conversation_id=conversation_id
            )
            
            if result["success"]:
                answer = result["answer"]
                confidence = result["confidence_score"]
                
                print(f"   ✅ Response generated (Confidence: {confidence:.3f})")
                
                # Show answer preview for context
                answer_preview = answer[:200] + "..." if len(answer) > 200 else answer
                print(f"   💬 Answer: {answer_preview}")
                
                conversation_results.append({
                    'turn': i,
                    'question': question,
                    'success': True,
                    'confidence': confidence,
                    'answer_length': len(answer)
                })
            else:
                print(f"   ❌ Failed: {result.get('error', 'Unknown error')}")
                conversation_results.append({
                    'turn': i,
                    'question': question,
                    'success': False,
                    'confidence': 0,
                    'answer_length': 0
                })
        
        except Exception as e:
            print(f"   ❌ Error: {str(e)}")
            conversation_results.append({
                'turn': i,
                'question': question,
                'success': False,
                'confidence': 0,
                'answer_length': 0
            })
        
        print()
    
    # Check conversation history
    try:
        history = rag_pipeline.llm_generator.get_conversation_history(limit=10)
        print(f"📚 Conversation History: {len(history)} messages stored")
        
        if history:
            print(f"   Most recent entries:")
            for entry in history[-4:]:
                role = entry.get('role', 'unknown')
                content_preview = entry.get('content', '')[:100] + "..." if len(entry.get('content', '')) > 100 else entry.get('content', '')
                print(f"      {role}: {content_preview}")
    
    except Exception as e:
        print(f"❌ Failed to retrieve conversation history: {str(e)}")
    
    conversation_success_rate = sum(1 for r in conversation_results if r['success']) / len(conversation_results) * 100
    print(f"✅ Conversation test completed. Success rate: {conversation_success_rate:.1f}%")
else:
    print("⚠️  Skipping conversation tests - pipeline not ready")
    conversation_results = []

## 5. RAG Performance Analysis

In [ ]:
# Analyze RAG performance
if rag_results:
    print("📊 Analyzing RAG performance...\n")
    
    # Convert to DataFrame
    df_rag = pd.DataFrame(rag_results)
    
    # Filter successful results for performance analysis
    successful_results = df_rag[df_rag['success'] == True]
    
    if len(successful_results) > 0:
        print(f"🎯 RAG Performance Metrics:")
        print(f"   Success rate: {(len(successful_results)/len(df_rag))*100:.1f}% ({len(successful_results)}/{len(df_rag)})")
        print(f"   Avg total time: {successful_results['query_time_ms'].mean():.1f}ms")
        print(f"   Avg retrieval time: {successful_results['retrieval_time'].mean():.1f}ms")
        print(f"   Avg generation time: {successful_results['generation_time'].mean():.1f}ms")
        print(f"   Avg confidence score: {successful_results['confidence'].mean():.3f}")
        print(f"   Avg answer length: {successful_results['answer_length'].mean():.0f} characters")
        print(f"   Avg topic coverage: {successful_results['topic_coverage'].mean():.2f} ({successful_results['topic_coverage'].mean()*100:.1f}%)")
        
        # Quality distribution
        high_quality = (successful_results['confidence'] >= 0.8).sum()
        medium_quality = ((successful_results['confidence'] >= 0.6) & (successful_results['confidence'] < 0.8)).sum()
        low_quality = (successful_results['confidence'] < 0.6).sum()
        
        print(f"\n🏆 Quality Distribution:")
        print(f"   High quality (≥0.8): {high_quality} ({high_quality/len(successful_results)*100:.1f}%)")
        print(f"   Medium quality (0.6-0.8): {medium_quality} ({medium_quality/len(successful_results)*100:.1f}%)")
        print(f"   Low quality (<0.6): {low_quality} ({low_quality/len(successful_results)*100:.1f}%)")
        
        # Time breakdown
        print(f"\n⏱️  Time Breakdown:")
        total_avg = successful_results['query_time_ms'].mean()
        retrieval_avg = successful_results['retrieval_time'].mean()
        generation_avg = successful_results['generation_time'].mean()
        
        print(f"   Retrieval: {retrieval_avg:.1f}ms ({retrieval_avg/total_avg*100:.1f}% of total)")
        print(f"   Generation: {generation_avg:.1f}ms ({generation_avg/total_avg*100:.1f}% of total)")
        print(f"   Other: {total_avg-retrieval_avg-generation_avg:.1f}ms ({(total_avg-retrieval_avg-generation_avg)/total_avg*100:.1f}% of total)")
    else:
        print(f"❌ No successful results to analyze")
else:
    print("⚠️  No RAG results available for analysis")
    df_rag = None

In [ ]:
# Visualize RAG performance
if df_rag is not None and len(df_rag) > 0 and len(df_rag[df_rag['success'] == True]) > 0:
    successful_df = df_rag[df_rag['success'] == True]
    
    # Set up plotting
    plt.style.use('default')
    sns.set_palette("husl")
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('RAG System Performance Analysis', fontsize=16, fontweight='bold')
    
    # 1. Confidence score distribution
    axes[0, 0].hist(successful_df['confidence'], bins=10, alpha=0.7, edgecolor='black')
    axes[0, 0].axvline(successful_df['confidence'].mean(), color='red', linestyle='--', label='Mean')
    axes[0, 0].set_title('Confidence Score Distribution')
    axes[0, 0].set_xlabel('Confidence Score')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].legend()
    
    # 2. Time breakdown
    time_components = ['Retrieval', 'Generation', 'Other']
    retrieval_avg = successful_df['retrieval_time'].mean()
    generation_avg = successful_df['generation_time'].mean()
    total_avg = successful_df['query_time_ms'].mean()
    other_avg = total_avg - retrieval_avg - generation_avg
    
    time_values = [retrieval_avg, generation_avg, other_avg]
    axes[0, 1].pie(time_values, labels=time_components, autopct='%1.1f%%', startangle=90)
    axes[0, 1].set_title('Processing Time Breakdown')
    
    # 3. Answer length vs confidence
    scatter = axes[1, 0].scatter(successful_df['answer_length'], successful_df['confidence'], 
                                alpha=0.7, s=60)
    axes[1, 0].set_title('Answer Length vs Confidence Score')
    axes[1, 0].set_xlabel('Answer Length (characters)')
    axes[1, 0].set_ylabel('Confidence Score')
    
    # 4. Topic coverage vs confidence
    scatter = axes[1, 1].scatter(successful_df['topic_coverage'], successful_df['confidence'], 
                                alpha=0.7, s=60)
    axes[1, 1].set_title('Topic Coverage vs Confidence Score')
    axes[1, 1].set_xlabel('Topic Coverage Ratio')
    axes[1, 1].set_ylabel('Confidence Score')
    
    plt.tight_layout()
    plt.show()
    
    # Summary table
    print("\n📋 RAG Results Summary:")
    summary_df = df_rag[['question', 'success', 'confidence', 'retrieved_docs', 'citations_count', 'query_time_ms']].copy()
    summary_df['question'] = summary_df['question'].str[:50] + '...'
    summary_df.columns = ['Question', 'Success', 'Confidence', 'Docs', 'Citations', 'Time (ms)']
    print(summary_df.to_string(index=False))
else:
    print("⚠️  No successful RAG results available for visualization")

## 6. Advanced RAG Features Testing

In [ ]:
# Test advanced RAG features
if pipeline_ready:
    print("🚀 Testing advanced RAG features...\n")
    
    # Test 1: Batch processing
    print("📦 Test 1: Batch Query Processing")
    batch_questions = [
        "What are LangChain prompt templates?",
        "How do I use callbacks in LangChain?",
        "What is the purpose of output parsers?"
    ]
    
    try:
        batch_start = time.time()
        batch_results = rag_pipeline.batch_query(
            questions=batch_questions,
            conversation_id="batch_test"
        )
        batch_time = (time.time() - batch_start) * 1000
        
        successful_batch = [r for r in batch_results if r.get('success', False)]
        print(f"   ✅ Batch processed: {len(successful_batch)}/{len(batch_questions)} successful")
        print(f"   ⏰ Total batch time: {batch_time:.1f}ms")
        print(f"   📊 Avg time per query: {batch_time/len(batch_questions):.1f}ms")
        
        if successful_batch:
            avg_confidence = sum(r.get('confidence_score', 0) for r in successful_batch) / len(successful_batch)
            print(f"   🎯 Avg batch confidence: {avg_confidence:.3f}")
        
    except Exception as e:
        print(f"   ❌ Batch processing failed: {str(e)}")
    
    print()
    
    # Test 2: Custom retrieval parameters
    print("🎛️  Test 2: Custom Retrieval Parameters")
    test_question = "How do I work with vector databases in LangChain?"
    
    try:
        # Test with high threshold for quality
        high_quality_result = rag_pipeline.query(
            question=test_question,
            retrieval_params={
                "top_k": 3,
                "score_threshold": 0.8
            }
        )
        
        print(f"   High threshold (0.8): {high_quality_result['retrieved_documents']} docs, confidence: {high_quality_result['confidence_score']:.3f}")
        
        # Test with lower threshold for more results
        broad_result = rag_pipeline.query(
            question=test_question,
            retrieval_params={
                "top_k": 8,
                "score_threshold": 0.5
            }
        )
        
        print(f"   Lower threshold (0.5): {broad_result['retrieved_documents']} docs, confidence: {broad_result['confidence_score']:.3f}")
        
    except Exception as e:
        print(f"   ❌ Custom parameters test failed: {str(e)}")
    
    print()
    
    # Test 3: Document type filtering
    print("📋 Test 3: Document Type Filtering")
    filter_question = "Show me examples of using LangChain chains"
    
    try:
        # Test with notebook filtering (likely to have examples)
        notebook_result = rag_pipeline.query(
            question=filter_question,
            retrieval_params={
                "document_types": ["notebooks", "python_files"]
            }
        )
        
        print(f"   Notebook/Python filter: {notebook_result['retrieved_documents']} docs")
        print(f"   Confidence: {notebook_result['confidence_score']:.3f}")
        
        # Show citation types if available
        if notebook_result.get('citations'):
            doc_types = [c.get('doc_type', 'unknown') for c in notebook_result['citations']]
            print(f"   Citation types: {', '.join(set(doc_types))}")
        
    except Exception as e:
        print(f"   ❌ Document filtering test failed: {str(e)}")
    
    print()

else:
    print("⚠️  Skipping advanced feature tests - pipeline not ready")

## 7. FastAPI RAG Endpoints Testing

In [ ]:
# Test FastAPI RAG endpoints
API_BASE_URL = "http://localhost:8000"

print("🌐 Testing FastAPI RAG endpoints...\n")

def test_endpoint(url, method='GET', data=None, timeout=60):
    """Test an API endpoint with extended timeout for RAG."""
    try:
        if method == 'GET':
            response = requests.get(url, timeout=timeout)
        elif method == 'POST':
            response = requests.post(url, json=data, timeout=timeout)
        
        return {
            'success': response.status_code == 200,
            'status_code': response.status_code,
            'data': response.json() if response.status_code == 200 else None,
            'error': response.text if response.status_code != 200 else None
        }
    except requests.exceptions.RequestException as e:
        return {
            'success': False,
            'status_code': None,
            'data': None,
            'error': str(e)
        }

# Test RAG health endpoint
print("🧪 Testing RAG health endpoint...")
health_result = test_endpoint(f'{API_BASE_URL}/api/v1/rag/health')

if health_result['success']:
    health_data = health_result['data']
    print(f"   ✅ RAG Health: {health_data.get('overall_health', 'unknown')}")
    
    components = health_data.get('components', {})
    for component, status in components.items():
        component_health = status.get('status', 'unknown')
        print(f"      {component}: {'✅' if component_health == 'healthy' else '❌'} {component_health}")
    
    api_server_ready = health_data.get('overall_health') == 'healthy'
else:
    print(f"   ❌ RAG Health check failed: {health_result.get('error', 'Unknown error')}")
    if 'Connection' in str(health_result.get('error', '')):
        print(f"      💡 Hint: Start the FastAPI server with: python app/main.py")
    api_server_ready = False

print()

In [ ]:
# Test RAG query endpoint
if api_server_ready:
    print("🧪 Testing RAG query endpoint...")
    
    rag_query_data = {
        "question": "What is LangChain and how do I get started?",
        "conversation_mode": "single",
        "use_citations": True,
        "generate_followup": True,
        "top_k": 5
    }
    
    query_start = time.time()
    rag_result = test_endpoint(
        f'{API_BASE_URL}/api/v1/rag/query',
        method='POST',
        data=rag_query_data,
        timeout=60
    )
    api_query_time = (time.time() - query_start) * 1000
    
    if rag_result['success']:
        data = rag_result['data']
        print(f"   ✅ RAG Query successful (API time: {api_query_time:.1f}ms)")
        print(f"      Question: {data.get('question', 'Unknown')[:50]}...")
        print(f"      Success: {data.get('success', False)}")
        print(f"      Confidence: {data.get('confidence_score', 0):.3f}")
        print(f"      Retrieved docs: {data.get('retrieved_documents', 0)}")
        print(f"      Citations: {len(data.get('citations', []))}")
        print(f"      Follow-ups: {len(data.get('follow_up_questions', []))}")
        
        # Show performance metrics
        performance = data.get('performance', {})
        print(f"      Performance: {performance.get('total_time_ms', 0):.1f}ms total")
        print(f"         Retrieval: {performance.get('retrieval_time_ms', 0):.1f}ms")
        print(f"         Generation: {performance.get('generation_time_ms', 0):.1f}ms")
        
        # Show answer preview
        answer = data.get('answer', '')
        answer_preview = answer[:200] + "..." if len(answer) > 200 else answer
        print(f"      Answer preview: {answer_preview}")
        
        # Show citations
        citations = data.get('citations', [])
        if citations:
            print(f"      Top citations:")
            for i, citation in enumerate(citations[:2], 1):
                source = citation.get('file_name', 'Unknown')
                score = citation.get('similarity_score', 0)
                print(f"         {i}. {source} (score: {score:.3f})")
        
        api_rag_test_passed = True
    else:
        print(f"   ❌ RAG Query failed: {rag_result.get('error', 'Unknown error')}")
        api_rag_test_passed = False
    
    print()

else:
    print("⚠️  Skipping RAG endpoint tests - server not ready")
    api_rag_test_passed = False

In [ ]:
# Test additional RAG endpoints
if api_server_ready:
    print("🧪 Testing additional RAG endpoints...\n")
    
    # Test batch RAG endpoint
    print("📦 Testing batch RAG endpoint...")
    batch_data = {
        "questions": [
            "What are LangChain document loaders?",
            "How do I create custom chains?"
        ],
        "conversation_id": "api_batch_test"
    }
    
    batch_result = test_endpoint(
        f'{API_BASE_URL}/api/v1/rag/batch',
        method='POST',
        data=batch_data,
        timeout=120  # Longer timeout for batch
    )
    
    if batch_result['success']:
        data = batch_result['data']
        print(f"   ✅ Batch RAG successful")
        print(f"      Questions: {data.get('total_questions', 0)}")
        print(f"      Batch time: {data.get('batch_time_ms', 0):.1f}ms")
        
        summary = data.get('summary', {})
        print(f"      Successful: {summary.get('successful_queries', 0)}")
        print(f"      Success rate: {summary.get('success_rate', 0):.1f}%")
        print(f"      Avg confidence: {summary.get('avg_confidence_score', 0):.3f}")
    else:
        print(f"   ❌ Batch RAG failed: {batch_result.get('error', 'Unknown error')}")
    
    # Test RAG analytics endpoint
    print("\n📊 Testing RAG analytics endpoint...")
    analytics_result = test_endpoint(f'{API_BASE_URL}/api/v1/rag/analytics')
    
    if analytics_result['success']:
        data = analytics_result['data']
        print(f"   ✅ RAG Analytics successful")
        
        pipeline_stats = data.get('pipeline_stats', {})
        print(f"      Total queries: {pipeline_stats.get('total_queries', 0)}")
        print(f"      Success rate: {data.get('success_rate', 0):.1f}%")
        print(f"      Avg response time: {pipeline_stats.get('avg_response_time', 0):.1f}ms")
        
        generation_stats = data.get('generation_stats', {})
        print(f"      LLM success rate: {generation_stats.get('success_rate', 0):.1f}%")
    else:
        print(f"   ❌ RAG Analytics failed: {analytics_result.get('error', 'Unknown error')}")
    
    # Test model info endpoint
    print("\n🤖 Testing model info endpoint...")
    models_result = test_endpoint(f'{API_BASE_URL}/api/v1/rag/models')
    
    if models_result['success']:
        data = models_result['data']
        print(f"   ✅ Model info retrieved")
        
        llm_models = data.get('llm_models', {})
        print(f"      Current LLM: {llm_models.get('current', 'Unknown')}")
        print(f"      Available models: {', '.join(llm_models.get('available', []))}")
        print(f"      Embedding model: {llm_models.get('embedding_model', 'Unknown')}")
    else:
        print(f"   ❌ Model info failed: {models_result.get('error', 'Unknown error')}")
else:
    print("⚠️  Skipping additional RAG endpoint tests - server not ready")

## 8. RAG System Analytics

In [ ]:
# Get comprehensive RAG analytics
if pipeline_ready:
    print("📊 Comprehensive RAG System Analytics\n")
    
    try:
        analytics = rag_pipeline.get_pipeline_analytics()
        
        # Pipeline statistics
        pipeline_stats = analytics.get('pipeline_stats', {})
        print(f"🚀 Pipeline Statistics:")
        print(f"   Total queries: {pipeline_stats.get('total_queries', 0)}")
        print(f"   Successful responses: {pipeline_stats.get('successful_responses', 0)}")
        print(f"   Failed responses: {pipeline_stats.get('failed_responses', 0)}")
        print(f"   Success rate: {analytics.get('success_rate', 0):.1f}%")
        print(f"   Avg response time: {pipeline_stats.get('avg_response_time', 0):.1f}ms")
        print(f"   Avg retrieval time: {pipeline_stats.get('avg_retrieval_time', 0):.1f}ms")
        print(f"   Avg generation time: {pipeline_stats.get('avg_generation_time', 0):.1f}ms")
        print(f"   Avg context docs: {pipeline_stats.get('avg_context_docs', 0):.1f}")
        
        # Retrieval analytics
        retrieval_analytics = analytics.get('retrieval_analytics', {})
        collection_stats = retrieval_analytics.get('collection_stats', {})
        print(f"\n🔍 Retrieval Analytics:")
        print(f"   Collection: {collection_stats.get('collection_name', 'Unknown')}")
        print(f"   Documents: {collection_stats.get('points_count', 0):,}")
        print(f"   Vector dimension: {collection_stats.get('vector_dimension', 'Unknown')}")
        print(f"   Distance metric: {collection_stats.get('distance_metric', 'Unknown')}")
        
        # Generation statistics
        generation_stats = analytics.get('generation_stats', {})
        print(f"\n🤖 Generation Statistics:")
        print(f"   Model: {generation_stats.get('model_name', 'Unknown')}")
        print(f"   Total requests: {generation_stats.get('total_requests', 0)}")
        print(f"   Successful generations: {generation_stats.get('successful_responses', 0)}")
        print(f"   LLM success rate: {generation_stats.get('success_rate', 0):.1f}%")
        print(f"   Avg response time: {generation_stats.get('avg_response_time', 0):.1f}ms")
        print(f"   Total tokens used: {generation_stats.get('total_tokens_used', 0):,}")
        print(f"   Avg tokens per request: {generation_stats.get('avg_tokens_per_request', 0):.1f}")
        
        # Configuration
        config = analytics.get('configuration', {})
        print(f"\n⚙️  Current Configuration:")
        for section, settings in config.items():
            if isinstance(settings, dict):
                print(f"   {section}:")
                for key, value in settings.items():
                    print(f"      {key}: {value}")
        
    except Exception as e:
        print(f"❌ Failed to get analytics: {str(e)}")
else:
    print("⚠️  Skipping analytics - pipeline not ready")

## 9. Summary and Final Assessment

In [ ]:
# Create comprehensive summary
print("📋 Phase 3 TODO 2 - Complete RAG System Summary\n")

# System status
print(f"🔧 System Status:")
print(f"   RAG Pipeline: {'✅ Ready' if pipeline_ready else '❌ Not ready'}")
print(f"   LLM Integration: {'✅ Working' if pipeline_ready else '❌ Not working'}")
print(f"   FastAPI Server: {'✅ Running' if api_server_ready else '❌ Not running'}")
print(f"   API RAG Endpoints: {'✅ Functional' if api_rag_test_passed else '❌ Issues detected'}")

# Performance summary
if rag_results and pipeline_ready:
    successful_queries = [r for r in rag_results if r['success']]
    if successful_queries:
        avg_confidence = sum(r['confidence'] for r in successful_queries) / len(successful_queries)
        avg_time = sum(r['query_time_ms'] for r in successful_queries) / len(successful_queries)
        avg_docs = sum(r['retrieved_docs'] for r in successful_queries) / len(successful_queries)
        
        print(f"\n📊 Performance Summary:")
        print(f"   Test queries: {len(rag_results)}")
        print(f"   Success rate: {len(successful_queries)/len(rag_results)*100:.1f}%")
        print(f"   Avg confidence: {avg_confidence:.3f}")
        print(f"   Avg response time: {avg_time:.1f}ms")
        print(f"   Avg docs retrieved: {avg_docs:.1f}")

# Conversation testing summary
if conversation_results:
    conv_success = sum(1 for r in conversation_results if r['success']) / len(conversation_results) * 100
    print(f"\n💬 Conversation Testing:")
    print(f"   Multi-turn conversation: {len(conversation_results)} turns")
    print(f"   Conversation success rate: {conv_success:.1f}%")
    print(f"   Memory management: {'✅ Working' if conv_success > 50 else '⚠️  Issues'}")

print(f"\n🚀 Features Implemented:")
features = [
    "✅ Gemini LLM integration with structured prompts",
    "✅ Complete RAG pipeline (retrieval + generation)", 
    "✅ Automatic source citations and references",
    "✅ Confidence scoring and answer quality assessment",
    "✅ Follow-up question generation",
    "✅ Conversation memory and context management",
    "✅ Batch query processing",
    "✅ Custom retrieval and generation parameters",
    "✅ FastAPI RAG endpoints with comprehensive models",
    "✅ Performance monitoring and analytics",
    "✅ Health checks and error handling"
]

for feature in features:
    print(f"   {feature}")

# Final status
overall_success = pipeline_ready and (not rag_results or len([r for r in rag_results if r['success']]) > 0)

print(f"\n🎯 Phase 3 TODO 2 Status: {'✅ COMPLETED' if overall_success else '⚠️  NEEDS ATTENTION'}")

if overall_success:
    print(f"\n🎉 Phase 3 Complete - RAG System Fully Operational!")
    print(f"\n🚀 Ready for Phase 4: Optimization & Enhancement")
    print(f"   Next focus: HyDE implementation and hybrid search")
    print(f"   Advanced techniques: Re-ranking and query expansion")
else:
    print(f"\n⚠️  Complete the following before proceeding:")
    if not pipeline_ready:
        print(f"   1. Ensure all dependencies are installed")
        print(f"   2. Verify Google API key configuration")
        print(f"   3. Check vector database connection")
    if rag_results and len([r for r in rag_results if r['success']]) == 0:
        print(f"   4. Debug RAG query failures")
        print(f"   5. Check retrieval and generation components")
    print(f"   6. Re-run this notebook after fixes")

# Show key API endpoints
print(f"\n🌐 Key RAG API Endpoints:")
print(f"   POST /api/v1/rag/query - Main RAG query endpoint")
print(f"   POST /api/v1/rag/batch - Batch query processing")
print(f"   GET /api/v1/rag/analytics - System analytics")
print(f"   GET /api/v1/rag/health - Health monitoring")
print(f"   Documentation: http://localhost:8000/docs")

## Next Steps

If Phase 3 TODO 2 completed successfully, you now have a **fully functional RAG system**:

### ✅ Completed Features:
1. **Complete RAG Pipeline**: Seamless integration of retrieval and generation
2. **LLM Integration**: Gemini 2.0 Flash with structured prompts and citations
3. **Quality Assessment**: Confidence scoring and answer validation
4. **Conversation Memory**: Multi-turn conversations with context preservation
5. **Advanced Features**: Batch processing, custom parameters, filtering
6. **Production API**: FastAPI endpoints with comprehensive documentation
7. **Monitoring & Analytics**: Performance tracking and health monitoring

### 🎯 RAG System Capabilities:
- **Question Answering**: Comprehensive answers with source citations
- **Context Awareness**: Multi-turn conversations with memory
- **Quality Control**: Confidence scoring and relevance assessment
- **Flexibility**: Customizable retrieval and generation parameters
- **Performance**: Optimized for speed and accuracy
- **Reliability**: Robust error handling and health monitoring

### 🚀 Next Phase:
**Phase 4: Optimization & Enhancement**
- **HyDE Implementation**: Hypothetical Document Embeddings for better retrieval
- **Hybrid Search**: Combining semantic and keyword search (BM25)
- **Re-ranking**: Advanced result refinement techniques
- **Query Enhancement**: Query expansion and reformulation

### 🌟 System Ready for Production Use!
Your RAG system is now capable of:
- Answering complex LangChain questions with citations
- Maintaining conversation context across multiple turns
- Processing batch queries efficiently
- Providing confidence scores for answer quality
- Generating relevant follow-up questions
- Operating through a production-ready API

**The core RAG implementation is complete and functional!**